In [1]:
import pandas as pd
import numpy as np

def run_feature_engineering_pipeline(input_path: str, output_path: str) -> pd.DataFrame:
    print("[+] Initiating Module 2: Temporal Feature Engineering Pipeline...")
    
    # 1. Load Cleaned Dataset from Module 1
    df = pd.read_csv(input_path, parse_dates=['Date'], low_memory=False)
    
    # Enforce chronological ordering per store
    df = df.sort_values(by=['Store', 'Date']).reset_index(drop=True)
    print(f"[+] Loaded Cleaned Schema: {len(df):,} rows.")
    
    # 2. Extract Calendar Components
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    df['Day'] = df['Date'].dt.day
    df['DayOfWeek'] = df['Date'].dt.dayofweek
    df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)
    
    # 3. Create Historical Lag Features (Grouped by Store)
    print("[+] Generating Historical Sales Lag Features...")
    for lag in [7, 14, 28, 365]:
        df[f'Sales_Lag_{lag}'] = df.groupby('Store')['Sales'].shift(lag)
        
    # 4. Create Rolling Moving Window Statistics
    # Target leakage avoid karne ke liye shift(1) lagana compulsory hai
    print("[+] Calculating Rolling Moving Means & Volatility (Standard Deviations)...")
    df['Sales_Rolling_Mean_7'] = df.groupby('Store')['Sales'].transform(lambda x: x.shift(1).rolling(7).mean())
    df['Sales_Rolling_Std_7'] = df.groupby('Store')['Sales'].transform(lambda x: x.shift(1).rolling(7).std())
    df['Sales_Rolling_Mean_30'] = df.groupby('Store')['Sales'].transform(lambda x: x.shift(1).rolling(30).mean())
    df['Sales_Rolling_Std_30'] = df.groupby('Store')['Sales'].transform(lambda x: x.shift(1).rolling(30).std())
    
    # 5. Interaction Terms
    df['Promo_x_DayOfWeek'] = df['Promo'] * df['DayOfWeek']
    
    # 6. Drop NaNs generated due to 365-day lag shifting
    initial_rows = len(df)
    df_engineered = df.dropna().reset_index(drop=True)
    print(f"[+] Dropped {initial_rows - len(df_engineered):,} NaN rows resulting from historical 365-day lags.")
    
    # 7. Save Feature-Engineered Dataset
    df_engineered.to_csv(output_path, index=False)
    print(f"[+] Engineered Dataset saved to '{output_path}'. Total Features: {df_engineered.shape[1]}, Total Rows: {len(df_engineered):,}")
    
    return df_engineered

# Execute Feature Engineering Pipeline
df_engineered = run_feature_engineering_pipeline("rossmann_cleaned.csv", "rossmann_engineered.csv")

C:\Users\reshu\AppData\Roaming\Python\Python312\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\reshu\AppData\Roaming\Python\Python312\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


[+] Initiating Module 2: Temporal Feature Engineering Pipeline...
[+] Loaded Cleaned Schema: 844,392 rows.
[+] Generating Historical Sales Lag Features...
[+] Calculating Rolling Moving Means & Volatility (Standard Deviations)...
[+] Dropped 631,722 NaN rows resulting from historical 365-day lags.
[+] Engineered Dataset saved to 'rossmann_engineered.csv'. Total Features: 31, Total Rows: 212,670
